In [52]:
# Below is the complete, ready-to-run notebook. Save as eeg_hybrid_crnn_fbgan.ipynb structure — each block is one cell. It is fully self-contained (Cells 1–9).

# Cell 1 — Environment Setup & Imports
# # ============================================================
# CELL 1: Environment Setup & Imports
# Paper: Zhang et al., Frontiers in Neuroscience, 2023
# Adapted to PhysioNet EEGMMIDB (109 subjects, C=64, T=640)
# ============================================================

import os
import glob
import copy
import random
import warnings

import numpy as np
import scipy.signal as sp_signal
from scipy.linalg import eig as generalized_eig

from sklearn.linear_model import Lasso
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report)

import matplotlib.pyplot as plt
import seaborn as sns

import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- GLOBAL ADAPTED PARAMETERS ----------------
DATA_ROOT = "/Users/ashokvarmabevara/MtechProj/eegmmidb"
NUM_SUBJECTS_TOTAL = 109            # S001..S109
NUM_SUBJECTS_TO_TEST = 10            # rapid-testing flag (default 5)
N_CHANNELS = 64                     # standard 10-10 montage
SFREQ = 160                         # Hz
TRIAL_SEC = 4.0
T_POINTS = int(SFREQ * TRIAL_SEC)   # T = 640
N_CLASSES = 4                       # L-fist, R-fist, both fists, both feet
TASK_RUNS = [4, 6, 8, 10, 12, 14]   # MI runs

# Preprocessing (paper)
BANDPASS_LO, BANDPASS_HI = 1.0, 38.0
BUTTER_ORDER = 5
SUB_BANDS = [(1, 4), (4, 8), (8, 12), (12, 16), (16, 20),
             (20, 24), (24, 28), (28, 32), (32, 35), (35, 38)]
N_CSP_EIGS_PER_CLASS = 4            # paper: 4 eigenvectors per sub-filter
TOTAL_CANDIDATES = len(SUB_BANDS) * N_CLASSES * N_CSP_EIGS_PER_CLASS  # = 160
# Loss / centroid shift (paper)
LAMBDA_CEN = 0.1
CENTROID_UPDATE_EVERY = 15          # epochs
ALPHA_SHIFT = 0.02                  # step size

# Training hyperparameters
NOISE_DIM = 100
BATCH_SIZE = 32
EPOCHS_CRNN = 30
EPOCHS_GAN = 200
FEAT_DIM = 128
LR_CRNN = 1e-3

print(f"Device: {DEVICE} | Input per sample: (1, {N_CHANNELS}, {T_POINTS})")


Device: cpu | Input per sample: (1, 64, 640)


In [53]:
# ============================================================
# CELL 2: Dimensionally adapted architecture diagrams
# (paper: C=22, T=1000  ->  here: C=64, T=640)
# ============================================================

CRNN_DF_DIAGRAM = r"""
=====================================================================
 CRNN-DF CLASSIFIER  (adapted: C=64, T=640)
=====================================================================
 Input (B, 1, 64, 640)
      |
      v
 Spatial Conv  kernel=(64, 45), stride=1      -> (B, 1, 1, 596)
      |
 Max Pool kernel=(1, 75), stride=10           -> (B, 1, 1, floor((596-75)/10)+1 = 53)
      |
 squeeze -> sequence (B, L=53, feat=1)
      |
 LSTM x2 layers, hidden_size = 64             -> last step (B, 64)
      |
 FC(64 -> 128) + ReLU + Dropout               -> discriminative feature v_i
      |
 FC(128 -> 4)                                 -> logits
      |
 Joint Loss = CE(logits) + lambda * CentralDistanceLoss(v_i)
=====================================================================
"""

FBGAN_DIAGRAM = r"""
=====================================================================
 FBGAN adapted to output (1, 64, 640)   [paper original: (1, 22, 1000)]
=====================================================================
 GENERATOR G(z): z ~ N(0,I), R^100
   Linear(100 -> 128*8*8) -> reshape (128, 8, 8)
   ConvT(128,64,k4,s2,p1)                 -> (64, 16, 16)  BN+ReLU
   ConvT(64,32,k4,s2,p1)                  -> (32, 32, 32)  BN+ReLU
   ConvT(32,16,k(1,5),s(1,5))             -> (16, 32, 160) BN+ReLU
   ConvT(16,1, k(2,4), s(2,4)) Tanh       -> (1, 64, 640)

 D_phi (SPATIAL discriminator):
   Conv2d(1,32,kernel=(64,1))             <- spatial kernel adapted (22,1)->(64,1)
   -> (B,32,1,640); LeakyReLU(0.2)
   Conv2d(32,64,kernel=(1,15)); LeakyReLU -> (B,64,1,626)
   AdaptiveMaxPool1d(25)                  <- scaled for T=640 (was ~39 for 1000)
   Flatten -> FC(64*25 -> 1) -> Sigmoid

 D_psi (TEMPORAL discriminator):
   Conv2d(1,32,kernel=(1,30))             -> (B,32,64,611)
   LeakyReLU(0.2); MaxPool2d((4,4),(4,4)) -> (B,32,16,152)
   Conv2(32,64,kernel=(1,15))            -> (B,64,16,138)
   AdaptiveMaxPool2d((4,25))
   Flatten -> FC(64*4*25 -> 1) -> Sigmoid

 D_total(x) = mean(D_phi(x)) + mean(D_psi(x))
=====================================================================
"""
print(CRNN_DF_DIAGRAM)
print(FBGAN_DIAGRAM)


 CRNN-DF CLASSIFIER  (adapted: C=64, T=640)
 Input (B, 1, 64, 640)
      |
      v
 Spatial Conv  kernel=(64, 45), stride=1      -> (B, 1, 1, 596)
      |
 Max Pool kernel=(1, 75), stride=10           -> (B, 1, 1, floor((596-75)/10)+1 = 53)
      |
 squeeze -> sequence (B, L=53, feat=1)
      |
 LSTM x2 layers, hidden_size = 64             -> last step (B, 64)
      |
 FC(64 -> 128) + ReLU + Dropout               -> discriminative feature v_i
      |
 FC(128 -> 4)                                 -> logits
      |
 Joint Loss = CE(logits) + lambda * CentralDistanceLoss(v_i)


 FBGAN adapted to output (1, 64, 640)   [paper original: (1, 22, 1000)]
 GENERATOR G(z): z ~ N(0,I), R^100
   Linear(100 -> 128*8*8) -> reshape (128, 8, 8)
   ConvT(128,64,k4,s2,p1)                 -> (64, 16, 16)  BN+ReLU
   ConvT(64,32,k4,s2,p1)                  -> (32, 32, 32)  BN+ReLU
   ConvT(32,16,k(1,5),s(1,5))             -> (16, 32, 160) BN+ReLU
   ConvT(16,1, k(2,4), s(2,4)) Tanh       -> (1, 64, 640)

 

In [54]:
# ============================================================
# CELL 3: Local offline loading from DATA_ROOT/S###/S###R##.edf
# Runs: 4 (task1 L-fist only), 6/10 (alternate L/R fists),
#       8 (task2 L-fist only), 12/14 (alternate fists vs feet)
# Labels: 0=L Fist, 1=R Fist, 2=Both Fists, 3=Both Feet
# ============================================================

def load_subject_edf(subject_id: int):
    """Return (X: (n,64,640) float32, y: (n,) int) for one subject."""
    sub_dir = os.path.join(DATA_ROOT, f"S{subject_id:03d}")
    if not os.path.isdir(sub_dir):
        return None, None

    wanted = {f"S{subject_id:03d}R{r:02d}.edf" for r in TASK_RUNS}
    paths = [p for p in sorted(glob.glob(os.path.join(sub_dir, "*.edf")))
             if os.path.basename(p) in wanted]

    X_list, y_list = [], []
    for path in paths:
        try:
            raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
            # Anomalous-sampling-rate guard: force 160 Hz so time axis == 640
            if abs(raw.info["sfreq"] - SFREQ) > 1e-6:
                raw.resample(SFREQ, verbose=False)
            raw.filter(BANDPASS_LO, BANDPASS_HI, method="iir",
                       iir_params={"order": BUTTER_ORDER,
                                   "btype": "bandpass", "ftype": "butter"},
                       verbose=False)
            raw.pick_types(eeg=True, meg=False, stim=False, eog=False,
                           misc=False, verbose=False)
            if len(raw.ch_names) != N_CHANNELS:
                continue                      # skip non-standard montages
            events, event_id = mne.events_from_annotations(raw, verbose=False)

            run_num = int(os.path.basename(path)[-6:-4])
            alternating = run_num in (6, 10, 12, 14)
            feet_run = run_num in (12, 14)

            epochs = mne.Epochs(raw, events, event_id=event_id,
                                tmin=0.0, tmax=TRIAL_SEC - 1.0 / SFREQ,
                                baseline=None, preload=True, verbose=False)
            data = epochs.get_data()
            for i in range(len(data)):
                d = data[i]
                if d.shape[1] < T_POINTS:     # strict (64, 640) guarantee
                    d = np.pad(d, ((0, 0), (0, T_POINTS - d.shape[1])))
                d = d[:, :T_POINTS].astype(np.float32)
                if not alternating:
                    label = 0
                elif feet_run:
                    label = 2 if i % 2 == 0 else 3
                else:
                    label = 0 if i % 2 == 0 else 1
                X_list.append(d)
                y_list.append(label)
        except Exception as exc:
            print(f"  [WARN] skipping {os.path.basename(path)}: {exc}")
    if not X_list:
        return None, None
    return np.stack(X_list), np.array(y_list)


def build_dataset(num_subjects: int = NUM_SUBJECTS_TO_TEST):
    Xs, ys, sids = [], [], []
    for sid in range(1, num_subjects + 1):
        print(f"Loading S{sid:03d} ...", end=" ")
        X, y = load_subject_edf(sid)
        if X is None or y is None:
            print("[no valid trials]")
            continue
        print(f"{X.shape[0]} trials")
        Xs.append(X); ys.append(y); sids.append(np.full(len(y), sid))
    return (np.concatenate(Xs), np.concatenate(ys),
            np.concatenate(sids))


X_all, y_all, sid_all = build_dataset(NUM_SUBJECTS_TO_TEST)
print("\nTotal:", X_all.shape, "| class counts:", np.bincount(y_all))

Loading S001 ... 180 trials
Loading S002 ... 180 trials
Loading S003 ... 180 trials
Loading S004 ... 180 trials
Loading S005 ... 180 trials
Loading S006 ... 180 trials
Loading S007 ... 180 trials
Loading S008 ... 180 trials
Loading S009 ... 180 trials
Loading S010 ... 180 trials

Total: (1800, 64, 640) | class counts: [900 300 300 300]


In [55]:
# ============================================================
# CELL 4: Preprocessing & FBCSP + LASSO module.
# Covariance (paper Eq.): R_c = (1/N_c) sum_i X_c X_c^T / trace(X_c X_c^T)
# OVR-CSP: 10 sub-bands x 4 classes, 4 eigenvectors per filter
# => 16 features per band => 160 candidate dimensions total.
# ============================================================

def zscore_standardize(X_train, X_test):
    """X' = (X - mu)/sigma computed on TRAIN set only."""
    mu = X_train.mean(axis=(0, 2), keepdims=True)
    sigma = X_train.std(axis=(0, 2), keepdims=True) + 1e-8
    return ((X_train - mu) / sigma).astype(np.float32), \
           ((X_test - mu) / sigma).astype(np.float32)


def _band_filter(X, lo, hi):
    sos = sp_signal.butter(BUTTER_ORDER, [lo, hi], btype="bandpass",
                           fs=SFREQ, output="sos")
    return sp_signal.sosfiltfilt(sos, X.astype(np.float64), axis=2)


def _ovr_csp_filters(X_band, y_band, n_comp=16):
    """Learn CSP filters per class (one-vs-rest). Returns list of W (C, n_comp)."""
    n_ch = X_band.shape[1]

    def cov(Xc):
        R = np.zeros((n_ch, n_ch))
        for xi in Xc:
            xi = xi @ xi.T
            R += xi / np.trace(xi)
        return R / len(Xc)

    filters = []
    half = n_comp // 2
    for c in range(N_CLASSES):
        pos, neg = X_band[y_band == c], X_band[y_band != c]
        if len(pos) < 2 or len(neg) < 2:
            W = np.eye(n_ch)[:, :n_comp]
        else:
            # Generalized eigenproblem on composite covariance (paper Eq.)
            eigvals, eigvecs = generalized_eig(cov(pos), cov(pos) + cov(neg))
            order = np.argsort(eigvals.real)[::-1]
            W = np.real(eigvecs[:, order])
            W = np.hstack([W[:, :half], W[:, -half:]])   # first 2 + last 2 pairs
        filters.append(W)
    return filters


def fit_fbcsp_bank(X, y, sub_bands=SUB_BANDS):
    """Fit full FBCSP bank on training data.
    Per band: 4 classes x 4 eigenvectors = 16 features -> 160 total."""
    bank = []                                    # [(band_idx, class_idx, W)]
    for b_idx, (lo, hi) in enumerate(sub_bands):
        Xb = _band_filter(X, lo, hi)
        for c, W in enumerate(
                _ovr_csp_filters(Xb, y, n_comp=N_CSP_EIGS_PER_CLASS)):
            bank.append((b_idx, c, W))
    return bank

def transform_fbcsp(X, bank, sub_bands=SUB_BANDS):
    """Project through bank; log-variance features. Output (n, 160)."""
    feats = []
    for b_idx, (lo, hi) in enumerate(sub_bands):
        Xb = _band_filter(X, lo, hi)
        band_feats = []
        for (b_j, _, W) in bank:
            if b_j != b_idx:
                continue
            proj = np.einsum("cs,nct->nst", W, Xb)     # (n, 16, T)

            var = (proj ** 2).mean(axis=2)
            band_feats.append(np.log(var + 1e-12))
        feats.append(np.concatenate(band_feats, axis=1))
    F = np.concatenate(feats, axis=1).astype(np.float32)
    assert F.shape[1] == TOTAL_CANDIDATES, f"FBCSP dim {F.shape[1]} != 160"
    return F


def lasso_select_features(F_train, y_train, alpha=0.01):
    """Multiclass OVR LASSO (L1); returns kept feature indices."""
    coef_sum = np.zeros(F_train.shape[1])
    for c in range(N_CLASSES):
        las = Lasso(alpha=alpha, max_iter=5000)
        las.fit(F_train, (y_train == c).astype(int))
        coef_sum += np.abs(las.coef_)
    kept = np.where(coef_sum > 1e-8)[0]
    if len(kept) < 8:
        kept = np.argsort(coef_sum)[-min(40, len(coef_sum)):]
    return kept

In [56]:
# See diagram in Cell 2.

class Generator(nn.Module):
    """Outputs fake EEG of shape (B, 1, 64, 640)."""

    def __init__(self, noise_dim=NOISE_DIM, debug=False):
        super().__init__()
        self.debug = debug
        self.fc = nn.Linear(noise_dim, 128 * 8 * 8)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, kernel_size=(1, 5), stride=(1, 5)),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16, 1, kernel_size=(2, 4), stride=(2, 4)),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.net(self.fc(z).view(-1, 128, 8, 8))
        if self.debug:
            print(f"[G] {tuple(z.shape)} -> {tuple(x.shape)}")   # (B,1,64,640)
        return x


class SpatialDiscriminatorDphi(nn.Module):
    """D_phi: judges channel-topography realism. Kernel (64,1)."""

    def __init__(self, debug=False):
        super().__init__()
        self.debug = debug
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(64, 1)),      # (B,32,1,640)
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=(1, 15)),     # (B,64,1,626)
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveMaxPool2d((1, 25)),              # FIX: 2D pool -> (B,64,1,25)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 25, 1), nn.Sigmoid())

    def forward(self, x):
        f = self.features(x)
        out = self.classifier(f)
        if self.debug:
            print(f"[D_phi] {tuple(x.shape)} -> {tuple(f.shape)} -> {tuple(out.shape)}")
        return out


class TemporalDiscriminatorDpsi(nn.Module):
    """D_psi: judges temporal waveform realism."""

    def __init__(self, debug=False):
        super().__init__()
        self.debug = debug
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(1, 30)),      # (B,32,64,611)
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(kernel_size=(4, 4), stride=(4, 4)),   # (B,32,16,152)
            nn.Conv2d(32, 64, kernel_size=(1, 15)),     # (B,64,16,138)
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveMaxPool2d((4, 25)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 4 * 25, 1), nn.Sigmoid())

    def forward(self, x):
        f = self.features(x)
        out = self.classifier(f)
        if self.debug:
            print(f"[D_psi] {tuple(x.shape)} -> {tuple(f.shape)} -> {tuple(out.shape)}")
        return out


def train_fbgan(loader, epochs=EPOCHS_GAN, noise_dim=NOISE_DIM, log_every=50):
    G = Generator(noise_dim).to(DEVICE)
    dphi = SpatialDiscriminatorDphi().to(DEVICE)
    dpsi = TemporalDiscriminatorDpsi().to(DEVICE)
    opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(list(dphi.parameters()) + list(dpsi.parameters()),
                             lr=2e-4, betas=(0.5, 0.999))
    bce = nn.BCELoss()

    for ep in range(epochs):
        g_loss_val = d_loss_val = 0.0
        for real, _ in loader:
            real = real.to(DEVICE)
            bs = real.size(0)
            # ones, zeros = torch.ones(bs, 1, device=DEVICE), torch.zeros(bs, 1, device=DEVICE)
            # In train_fbgan:
            ones = torch.full((bs, 1), 0.9, device=DEVICE)   # label smoothing
            zeros = torch.zeros(bs, 1, device=DEVICE)
            # optionally add noise to labels:
            ones = ones + 0.05 * torch.rand_like(ones)
            zeros = zeros + 0.05 * torch.rand_like(zeros)

            opt_d.zero_grad()
            fake = G(torch.randn(bs, noise_dim, device=DEVICE)).detach()
            loss_d = 0.5 * (bce(dphi(real), ones) + bce(dphi(fake), zeros)
                            + bce(dpsi(real), ones) + bce(dpsi(fake), zeros))
            loss_d.backward(); opt_d.step()

            opt_g.zero_grad()
            fake = G(torch.randn(bs, noise_dim, device=DEVICE))
            loss_g = 0.5 * (bce(dphi(fake), ones) + bce(dpsi(fake), ones))
            loss_g.backward(); opt_g.step()

            g_loss_val, d_loss_val = loss_g.item(), loss_d.item()
        if (ep + 1) % log_every == 0:
            print(f"[FBGAN] ep {ep+1}/{epochs}  D={d_loss_val:.4f}  G={g_loss_val:.4f}")
    return G, dphi, dpsi

In [60]:
class CRNN_DF(nn.Module):
    """
    Hybrid CNN + 2-layer LSTM classifier with discriminative-feature head.

    Pipeline (adapted to C=64, T=640):
      Input (B,1,64,640)
      -> SpatialConv kernel (64,45)          -> (B,1,1,596)
      -> MaxPool (1,75) stride 10            -> (B,1,1,53)   seq_len L=53
      -> LSTM(2 layers, hidden=64)           -> (B,64)
      -> FC(64->128)+ReLU+Dropout            -> v_i  (discriminative feature)
      -> FC(128->4)                          -> logits
    """

    def __init__(self, n_channels=N_CHANNELS, n_classes=N_CLASSES,
                 lstm_hidden=64, feat_dim=FEAT_DIM, debug=False):
        super().__init__()
        self.debug = debug
        self.spatial_conv = nn.Conv2d(1, 1, kernel_size=(n_channels, 45), stride=1)
        self.pool = nn.MaxPool2d(kernel_size=(1, 75), stride=10)
        self.lstm = nn.LSTM(input_size=1, hidden_size=lstm_hidden,
                            num_layers=2, batch_first=True)
        self.feature_head = nn.Sequential(
            nn.Linear(lstm_hidden, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.classifier = nn.Linear(feat_dim, n_classes)

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)                        # (B,1,C,T)
        h = self.spatial_conv(x)                      # (B,1,1,596)
        h = self.pool(h)                              # (B,1,1,L)
        L = h.size(-1)
        h = h.squeeze(1).squeeze(1).unsqueeze(-1)     # (B,L,1)
        h, _ = self.lstm(h)                           # (B,L,64)
        v = self.feature_head(h[:, -1, :])            # (B,128)
        logits = self.classifier(v)
        if self.debug:
            print(f"[CRNN] input {tuple(x.shape)} -> conv {tuple(self.spatial_conv(x).shape)}"
                  f" -> pool ({L}) -> v_i {tuple(v.shape)} -> logits {tuple(logits.shape)}")
        return logits, v


# Quick shape sanity check (validates 1000 -> 640 adaptation)
_model = CRNN_DF(debug=True).to(DEVICE)
with torch.no_grad():
    _model(torch.randn(2, 64, T_POINTS, device=DEVICE))
del _model

[CRNN] input (2, 1, 64, 640) -> conv (2, 1, 1, 596) -> pool (53) -> v_i (2, 128) -> logits (2, 4)


In [61]:
class CenterShiftLoss(nn.Module):
    """Joint Loss = CE(logits) + lambda * L_cen.

    L_cen = (1/b) * sum_{i=1}^{b} || v_i^{k} - cen_{y_i}^{k} ||_2
    Centroid update every `update_every` epochs, step size alpha:
        cen_k <- cen_k + alpha * mean(v_i | y_i = k - cen_k)
    """

    def __init__(self, n_classes=N_CLASSES, feat_dim=FEAT_DIM,
                 lam=LAMBDA_CEN, alpha=ALPHA_SHIFT,
                 update_every=CENTROID_UPDATE_EVERY):
        super().__init__()
        self.lam = lam
        self.alpha = alpha
        self.update_every = update_every
        self.centroids = torch.randn(n_classes, feat_dim) * 0.01

    @torch.no_grad()
    def init_centroids(self, model, loader, device=DEVICE):
        model.eval()
        sums = torch.zeros_like(self.centroids, device=device)
        counts = torch.zeros(self.centroids.size(0), device=device)
        for x, y in loader:
            _, v = model(x.to(device))
            y = y.to(device)
            for c in range(counts.numel()):
                mask = y == c
                if mask.any():
                    sums[c] += v[mask].sum(dim=0)
                    counts[c] += mask.sum()
        self.centroids = sums / counts.clamp(min=1).unsqueeze(1)

    def central_distance_loss(self, v, y):
        cen = self.centroids.to(v.device)[y]
        return torch.norm(v - cen, p=2, dim=1).mean()

    def forward(self, logits, v, y, epoch=None):
        ce = F.cross_entropy(logits, y)
        l_cen = self.central_distance_loss(v, y)
        return ce + self.lam * l_cen, ce.item(), l_cen.item()

    @torch.no_grad()
    def maybe_shift_centroids(self, model, loader, epoch, device=DEVICE):
        """Called at end of each epoch; shifts centroids every 15 epochs."""
        if epoch % self.update_every != 0 or epoch == 0:
            return False
        model.eval()
        sums = torch.zeros_like(self.centroids, device=device)
        counts = torch.zeros(self.centroids.size(0), device=device)
        for x, y in loader:
            _, v = model(x.to(device))
            y = y.to(device)
            for c in range(counts.numel()):
                mask = y == c
                if mask.any():
                    sums[c] += v[mask].sum(dim=0)
                    counts[c] += mask.sum()
        new_means = sums / counts.clamp(min=1).unsqueeze(1)
        self.centroids = self.centroids.to(device) \
            + self.alpha * (new_means - self.centroids.to(device))
        return True

In [62]:
# ============================================================
# CELL 8: Leave-One-Subject-Out over subjects 1..NUM_SUBJECTS_TO_TEST.
# For each held-out subject: fit z-score/CSP/LASSO on train subjects,
# train CRNN-DF with joint loss, augment training set with FBGAN samples.
# ============================================================

def make_loader(X, y, shuffle=True, batch_size=BATCH_SIZE):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32).unsqueeze(1),   # (n,1,64,640)
        torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


from sklearn.linear_model import LogisticRegression
import torch.nn.functional as Fn

from sklearn.linear_model import LogisticRegression
import torch.nn.functional as Fn


def train_one_fold(train_sids, test_sid, use_gan=True, epochs=EPOCHS_CRNN):
    # ---------- Split by subject ----------
    tr_mask = ~np.isin(sid_all, [test_sid])
    te_mask = sid_all == test_sid

    X_tr_raw, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_te_raw, y_te = X_all[te_mask], y_all[te_mask]

    # ---------- Preprocessing fitted on TRAIN subjects only ----------
    X_tr, X_te = zscore_standardize(X_tr_raw, X_te_raw)
    bank = fit_fbcsp_bank(X_tr, y_tr)
    F_tr, F_te = transform_fbcsp(X_tr, bank), transform_fbcsp(X_te, bank)
    kept = lasso_select_features(F_tr, y_tr)
    print(f"  LASSO selected {len(kept)}/{F_tr.shape[1]} features")

    # ---------- Parallel FBCSP head on LASSO-selected features ----------
    fbcsp_clf = LogisticRegression(max_iter=2000, C=1.0)
    fbcsp_clf.fit(F_tr[:, kept], y_tr)

    loader_tr = make_loader(X_tr, y_tr)
    loader_te = make_loader(X_te, y_te, shuffle=False)

    # ---------- Optional FBGAN augmentation ----------
    if use_gan:
        print("  Training FBGAN ...")
        G, _, _ = train_fbgan(loader_tr, epochs=EPOCHS_GAN)
        G.eval()
        n_aug = min(len(X_tr), 500)
        with torch.no_grad():
            fake = G(torch.randn(n_aug, NOISE_DIM,
                                 device=DEVICE)).cpu().squeeze(1)
        y_fake = np.arange(n_aug) % N_CLASSES
        X_tr = np.concatenate(
            [X_tr, ((fake.numpy() + 1.0) / 2.0).astype(np.float32)])
        y_tr = np.concatenate([y_tr, y_fake])
        loader_tr = make_loader(X_tr, y_tr)
        print(f"  Augmented with {n_aug} synthetic samples")

    # ---------- Train CRNN-DF ----------
    model = CRNN_DF().to(DEVICE)
    criterion = CenterShiftLoss().to(DEVICE)
    criterion.init_centroids(model, loader_tr)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_CRNN)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                           T_max=epochs)

    for epoch in range(epochs):
        model.train()
        tot_loss = tot_ce = tot_cen = 0.0
        for x, y in loader_tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits, v = model(x)
            loss, ce_v, cen_v = criterion(logits, v, y, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            tot_loss += loss.item()
            tot_ce += ce_v
            tot_cen += cen_v
        shifted = criterion.maybe_shift_centroids(model, loader_tr, epoch + 1)
        scheduler.step()
        if (epoch + 1) % 5 == 0:
            print(f"  ep {epoch+1:02d}: loss={tot_loss/len(loader_tr):.4f} "
                  f"(CE={tot_ce/len(loader_tr):.4f}, "
                  f"Lcen={tot_cen/len(loader_tr):.4f})"
                  f"{' [centroids shifted]' if shifted else ''}")

    # ---------- Evaluate: CRNN softmax + FBCSP head, late fusion ----------
    model.eval()
    preds, labels, feats = [], [], []
    with torch.no_grad():
        for x, y in loader_te:
            logits, v = model(x.to(DEVICE))
            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(y.numpy())
            feats.append(v.cpu())
    labels = np.array(labels)
    preds = np.array(preds)
    feats_np = torch.cat(feats).numpy()

    probs_nn = Fn.softmax(
        torch.tensor([0]), dim=0)[:0].new_zeros(1)  # placeholder removed below
    # Compute NN probabilities properly:
    nn_probs_list = []
    with torch.no_grad():
        for x, _ in loader_te:
            logits, _ = model(x.to(DEVICE))
            nn_probs_list.append(Fn.softmax(logits, dim=1).cpu().numpy())
    probs_nn = np.concatenate(nn_probs_list, axis=0)

    probs_fbcsp = fbcsp_clf.predict_proba(F_te[:, kept])
    fused_probs = 0.5 * probs_nn + 0.5 * probs_fbcsp
    fused_preds = fused_probs.argmax(axis=1)

    acc_crnn = accuracy_score(labels, preds)
    acc_fused = accuracy_score(labels, fused_preds)
    print(f"  ==> S{test_sid:03d}  CRNN-only: {acc_crnn:.4f} | "
          f"fused CRNN+FBCSP: {acc_fused:.4f}\n")
    return acc_fused, fused_preds, labels, feats_np



results = {}
for sid in range(1, NUM_SUBJECTS_TO_TEST + 1):
    if sid not in set(sid_all.tolist()):
        continue
    print(f"=== LOSO fold: testing on S{sid:03d} ===")
    results[sid] = train_one_fold(None, sid, use_gan=True)

=== LOSO fold: testing on S001 ===
  LASSO selected 91/160 features
  Training FBGAN ...
[FBGAN] ep 50/200  D=0.7569  G=1.4837


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 9: Aggregate metrics + t-SNE of discriminative features v_i
# ============================================================

CLASS_NAMES = ["Left Fist", "Right Fist", "Both Fists", "Both Feet"]

accs = [r[0] for r in results.values()]
print("=" * 60)
print(f"LOSO mean accuracy over {len(accs)} subjects: "
      f"{np.mean(accs):.4f} +/- {np.std(accs):.4f}")
print("=" * 60)

# Confusion matrix pooled across folds
all_preds = np.concatenate([r[1] for r in results.values()])
all_labels = np.concatenate([r[2] for r in results.values()])

plt.figure(figsize=(7, 6))
sns.heatmap(confusion_matrix(all_labels, all_preds), annot=True, fmt="d",
            cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Pooled Confusion Matrix (LOSO)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.show()

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# Per-subject accuracy bar plot
plt.figure(figsize=(8, 4))
plt.bar([f"S{s:03d}" for s in results], [r[0] for r in results.values()],
        color="steelblue")
plt.axhline(np.mean(accs), ls="--", c="red", label=f"mean={np.mean(accs):.3f}")
plt.ylabel("Accuracy"); plt.title("Per-subject LOSO accuracy")
plt.legend(); plt.tight_layout(); plt.show()

# t-SNE on discriminative features from the first fold
first_sid = next(iter(results))
_, _, _, v_feats = results[first_sid]
v_labels = y_all[sid_all == first_sid][:len(v_feats)]

emb = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(v_feats)
plt.figure(figsize=(7, 6))
for c in range(N_CLASSES):
    m = v_labels == c
    plt.scatter(emb[m, 0], emb[m, 1], s=14, alpha=0.7, label=CLASS_NAMES[c])
plt.title(f"t-SNE of Discriminative Features v_i (held-out S{first_sid:03d})")
plt.legend(); plt.tight_layout(); plt.show()